Step 1: Load and prepare documents

In [2]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = ["https://lilianweng.github.io/posts/2023-06-23-agent/"]

# Load documents from the URLs
docs = [WebBaseLoader(
                url, 
                requests_kwargs={"headers": {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}}
            ).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

USER_AGENT environment variable not set, consider setting it to identify your requests.


Step 2: Split documents into chunks

In [3]:
# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

Step 3: Create a vector store

## Environment Variables Setup (Required)

### Set in Terminal/PowerShell (Temporary - current session only)
```bash
# Windows PowerShell
$env:AZURE_OPENAI_API_KEY="your_api_key_here"
$env:AZURE_OPENAI_ENDPOINT="https://your-resource-name.openai.azure.com/"

# Windows Command Prompt
set AZURE_OPENAI_API_KEY=your_api_key_here
set AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com/

# Linux/macOS
export AZURE_OPENAI_API_KEY="your_api_key_here"
export AZURE_OPENAI_ENDPOINT="https://your-resource-name.openai.azure.com/"
```

In [ ]:
import os
load_dotenv()

print("Current environment variables:")
print(f"AZURE_OPENAI_API_KEY: {'SET' if os.getenv('AZURE_OPENAI_API_KEY') else 'NOT SET'}")
print(f"AZURE_OPENAI_ENDPOINT: {'SET' if os.getenv('AZURE_OPENAI_ENDPOINT') else 'NOT SET'}")
print(f"AZURE_OPENAI_API_VERSION: {'SET' if os.getenv('AZURE_OPENAI_API_VERSION') else 'NOT SET'}")

Current environment variables:
AZURE_OPENAI_API_KEY: SET
AZURE_OPENAI_ENDPOINT: SET
AZURE_OPENAI_API_VERSION: SET


In [13]:
# ALTERNATIVE: Load environment variables from a .env file
# This is the most secure and convenient way for development

try:
    from dotenv import load_dotenv
    load_dotenv()  # Load .env file from current directory
    print("✅ Loaded environment variables from .env file")
except ImportError:
    print("📦 python-dotenv not installed. Install with: pip install python-dotenv")
    print("💡 Or set environment variables manually in your shell")
except Exception as e:
    print(f"⚠️  Could not load .env file: {e}")
    print("💡 Make sure .env file exists or set environment variables manually")

✅ Loaded environment variables from .env file


In [22]:
import os
from langchain_community.vectorstores import SKLearnVectorStore  
from langchain_openai import AzureOpenAIEmbeddings

# Azure OpenAI Configuration from environment variables
azure_openai_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')
azure_openai_api_key = os.getenv('AZURE_OPENAI_API_KEY')
azure_openai_chat_deployment = os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT', 'gpt-4o-mini')
azure_openai_api_version = os.getenv('AZURE_OPENAI_API_VERSION', '2024-08-01-preview')
azure_openai_embedding_deployment = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-small')

vectorstore = SKLearnVectorStore.from_documents(
            documents=doc_splits,
            embedding=AzureOpenAIEmbeddings(
                azure_endpoint=azure_openai_endpoint,
                azure_deployment=azure_openai_embedding_deployment,
                openai_api_key=azure_openai_api_key,
                openai_api_version=azure_openai_api_version,
            ),
        )
      
retriever = vectorstore.as_retriever(k=4)
print(f"\n✅ Vector store created successfully!")


✅ Vector store created successfully!


Step 4: Set up the LLM and prompt template

In [23]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

# Define the prompt template for the LLM
prompt = PromptTemplate(
    template="""You are an assistant for question-answering tasks.
    Use the following documents to answer the question.
    If you don't know the answer, just say that you don't know.
    Use three sentences maximum and keep the answer concise:
    Question: {question}
    Documents: {documents}
    Answer:
    """,
    input_variables=["question", "documents"],
)

llm = AzureChatOpenAI(
    azure_endpoint=azure_openai_endpoint,
    azure_deployment=azure_openai_chat_deployment,
    openai_api_key=azure_openai_api_key,
    openai_api_version=azure_openai_api_version,
    temperature=0,
)
# Create a chain combining the prompt template and LLM
rag_chain = prompt | llm | StrOutputParser()

# Equivalent explicit forms: 
# rag_chain = RunnableSequence(prompt, llm, StrOutputParser())
# rag_chain = prompt.chain(llm).chain(StrOutputParser())


Step 5: Integrate the retriever and LLM into a RAG application

In [24]:
# Define the RAG application class
class RAGApplication:
    def __init__(self, retriever, rag_chain):
        self.retriever = retriever
        self.rag_chain = rag_chain
        
    def run(self, question):
        # Retrieve relevant documents
        documents = self.retriever.invoke(question)
        # Extract content from retrieved documents
        doc_texts = "\\n".join([doc.page_content for doc in documents])
        # Get the answer from the language model
        answer = self.rag_chain.invoke({"question": question, "documents": doc_texts})
        return answer

# Check if both components were set up successfully
if 'vectorstore' in locals() and vectorstore is not None and 'rag_chain' in locals() and 'rag_chain' in locals():
    print("✅ All components ready! RAG Application can be created.")
else:
    print("🚨 Cannot create RAG application - some components failed to initialize.")
    if 'vectorstore' not in locals() or vectorstore is None:
        print("   - Vector store creation failed")
    if 'rag_chain' not in locals():
        print("   - RAG chain creation failed")
    print("\nPlease fix the Azure OpenAI configuration issues above before proceeding.")

✅ All components ready! RAG Application can be created.


Step 6: Test your RAG application

In [25]:
# Create and test the RAG application
if 'vectorstore' in locals() and vectorstore is not None and 'rag_chain' in locals():
    rag_app = RAGApplication(retriever, rag_chain)

    # Test with some questions
    questions = [
        "What are the key components of an AI agent?",
        "How do adversarial attacks work on large language models?"
    ]

    # Test each question
    for question in questions:
        print(f"\n🤖 Question: {question}")
        print("-" * 50)
        try:
            answer = rag_app.run(question)
            print(f"📋 Answer: {answer}")
        except Exception as e:
            print(f"❌ Error: {e}")
        print("=" * 50)
else:
    print("🚨 Cannot test RAG application - setup incomplete.")
    print("Please run the previous cells to fix configuration issues.")


🤖 Question: What are the key components of an AI agent?
--------------------------------------------------
📋 Answer: The key components of an AI agent are Planning (including task decomposition and self-reflection), Memory (short-term and long-term memory for learning and information retention), and Tool Use (calling external APIs for additional information and capabilities). The LLM acts as the agent’s brain, coordinating these components to solve complex tasks efficiently.

🤖 Question: How do adversarial attacks work on large language models?
--------------------------------------------------
📋 Answer: The provided documents do not contain information on how adversarial attacks work on large language models. Therefore, I don't know the answer based on these sources.


In [26]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Better RAG chain: retrieval + formatting + generation in one LCEL pipeline
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

better_prompt = PromptTemplate(
    template="""You are a precise QA assistant.
Use only the context below to answer the question.
If the answer is not in the context, say "I don't know."
Keep the response to at most 3 sentences.

Question: {question}
Context:
{context}

Answer:""",
    input_variables=["question", "context"],
)

better_rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever | RunnableLambda(format_docs),
    }
    | better_prompt
    | llm
    | StrOutputParser()
)

# Quick test
print(better_rag_chain.invoke("What are the key components of an AI agent?"))

The key components of an AI agent are Planning (including subgoal decomposition and self-reflection), Memory (short-term and long-term memory), and Tool use (calling external APIs for additional information and capabilities). The LLM acts as the agent’s brain, coordinating these components.
